In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub


from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold

import warnings
warnings.filterwarnings('ignore')


In [ ]:
data_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(data_path)

print(f"Shape: {df_food.shape}")


In [ ]:
df_food.head()

In [ ]:
df_food.info()

In [ ]:
df_food.describe()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=20, edgecolor='black')
plt.title('Delivery Time Disribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
df_food.drop(columns=['Order_ID'])


In [ ]:
#dropping rows with missing values
df_food.isnull().sum()

missing_values = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs']

df_clean = df_food.dropna(subset=missing_values).copy()
print(f"Shape after cleaning: {df_clean.shape}")

#filling missing target values with mean
df_clean['Delivery_Time'] = df_clean['Delivery_Time'].fillna(df_clean['Delivery_Time'].mean())


In [ ]:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))



df_clean.head()


In [ ]:

scaler = StandardScaler()
feature_cols = df_clean[['Distance_km','Weather','Traffic_Level','Time_of_Day','Vehicle_Type','Preparation_Time_min','Courier_Experience_yrs']]
feature_cols = scaler.fit_transform(feature_cols)



In [ ]:
#we will use the same target distribution plot before
plt.figure(figsize=(10, 5))
plt.hist(df_clean['Delivery_Time'].dropna(), bins=20, edgecolor='black')
plt.title('Delivery Time Disribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

#since the data isn't that skewed we could say its balanced
print("The Target data is balanced")

In [ ]:
X = df_clean[['Distance_km','Weather','Traffic_Level','Time_of_Day','Vehicle_Type','Preparation_Time_min','Courier_Experience_yrs']]
y = df_clean['Delivery_Time']

In [ ]:


# Define K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    # print shapes
    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)



# Train Random Forest Regressor
    model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    print("Model trained!")
    print("-" * 30)
# Predict and evaluate
    y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
print(f"MAE:  {mae:,.0f} mins")




In [ ]:
feature_importance = pd.DataFrame({
    'feature': ['Distance_km','Weather','Traffic_Level','Time_of_Day','Vehicle_Type','Preparation_Time_min','Courier_Experience_yrs'],
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--', linewidth=2)
plt.xlabel("Actual y_test (Ground Truth)")
plt.ylabel("Predicted y_pred (Linear Regression)")
plt.title("Linear Regression: Predictions vs. Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: